# 01 — Data Preprocessing

This notebook covers:
1. Corrupt image detection & removal
2. Class balance analysis
3. Building the Keras `ImageDataGenerator` pipeline
4. Visualising augmented samples
5. Final dataset summary

In [1]:
import os, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings('ignore')

ROOT       = r"D:\Hams Khaled\CS Year 3 Sem 2\Image Processing\Final Project\project"
SPLIT_DIR  = os.path.join(ROOT, "data", "split")
PLOTS_DIR  = os.path.join(ROOT, "results", "plots")
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

os.makedirs(PLOTS_DIR, exist_ok=True)
print("Paths configured.")
print(f"  SPLIT_DIR : {SPLIT_DIR}")
print(f"  PLOTS_DIR : {PLOTS_DIR}")


Paths configured.
  SPLIT_DIR : D:\Hams Khaled\CS Year 3 Sem 2\Image Processing\Final Project\project\data\split
  PLOTS_DIR : D:\Hams Khaled\CS Year 3 Sem 2\Image Processing\Final Project\project\results\plots


## Step 1 — Detect & Remove Corrupted Images

We iterate over every image in the split folders and attempt to open it with PIL. Any file that fails is logged and removed.

In [2]:
SPLITS  = ["train", "val", "test"]
CLASSES = ["fresh", "rotten"]

corrupted = []  
total_checked = 0

for split in SPLITS:
    for cls in CLASSES:
        folder = os.path.join(SPLIT_DIR, split, cls)
        if not os.path.isdir(folder):
            print(f"[WARN] Folder not found, skipping: {folder}")
            continue
        for fname in os.listdir(folder):
            fpath = os.path.join(folder, fname)
            total_checked += 1
            try:
                with Image.open(fpath) as img:
                    img.verify()
            except Exception as e:
                corrupted.append(fpath)
                print(f"  Corrupted: {fpath}  — {e}")

print(f"\n  Total images checked : {total_checked}")
print(f"  Corrupted files found: {len(corrupted)}")

# Remove corrupted files
for fpath in corrupted:
    os.remove(fpath)
    print(f"  Removed: {fpath}")

if not corrupted:
    print("No corrupted images found.")



  Total images checked : 13599
  Corrupted files found: 0
No corrupted images found.


## Step 2 — Class Balance Analysis

Count images per class per split and visualise as a bar chart.

In [3]:
counts = {}
for split in SPLITS:
    counts[split] = {}
    for cls in CLASSES:
        folder = os.path.join(SPLIT_DIR, split, cls)
        n = len(os.listdir(folder)) if os.path.isdir(folder) else 0
        counts[split][cls] = n

print(f"{'Split':<8}", end="")
for cls in CLASSES:
    print(f"  {cls:<10}", end="")
print()
print("-" * 32)
for split in SPLITS:
    print(f"{split:<8}", end="")
    for cls in CLASSES:
        print(f"  {counts[split][cls]:<10}", end="")
    print()

total_fresh  = sum(counts[s]["fresh"]  for s in SPLITS)
total_rotten = sum(counts[s]["rotten"] for s in SPLITS)
total_all    = total_fresh + total_rotten
ratio = total_fresh / total_all * 100 if total_all else 0

print(f"\nTotal fresh : {total_fresh}")
print(f"Total rotten: {total_rotten}")
if abs(ratio - 50) > 10:
    print(f"WARNING: Imbalanced dataset! Fresh={ratio:.1f}%")
else:
    print(f"Class balance OK (fresh={ratio:.1f}%)")


Split     fresh       rotten    
--------------------------------
train     4132        5386      
val       886         1154      
test      886         1155      

Total fresh : 5904
Total rotten: 7695
Class balance OK (fresh=43.4%)


In [5]:
import pandas as pd

df = pd.DataFrame(counts).T

ax = df.plot(kind='bar', color=['#4CAF50', '#F44336'], figsize=(8, 5), width=0.7)

ax.set_title("Class Balance per Split")
ax.set_ylabel("Number of Images")
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(os.path.join(PLOTS_DIR, "class_balance_simple.png"), dpi=150)
out_path = os.path.join(PLOTS_DIR, "class_balance.png")
print(f"Class balance chart saved → {out_path}")

Class balance chart saved → D:\Hams Khaled\CS Year 3 Sem 2\Image Processing\Final Project\project\results\plots\class_balance.png


## Step 3 — ImageDataGenerator Pipeline

We build three generators:
- **Train**: resize + normalize + augmentation
- **Val / Test**: resize + normalize only

In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Train generator — WITH augmentation
train_datagen = ImageDataGenerator(
    rescale          = 1./255,
    horizontal_flip  = True,
    rotation_range   = 20,
    zoom_range       = 0.2,
    brightness_range = [0.8, 1.2],
)

# Val & Test generators — normalize only
val_datagen  = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, "train"),
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = "binary",
    shuffle     = True,
)

val_generator = val_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, "val"),
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = "binary",
    shuffle     = False,
)

test_generator = test_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, "test"),
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = "binary",
    shuffle     = False,
)

print("Generators built successfully.")
print(f"Class indices: {train_generator.class_indices}")


Found 9518 images belonging to 2 classes.
Found 2040 images belonging to 2 classes.
Found 2041 images belonging to 2 classes.
Generators built successfully.
Class indices: {'fresh': 0, 'rotten': 1}


## Step 4 — Visualise Augmented Samples

Display a 3×3 grid of randomly augmented training images.

In [9]:
import math

imgs, labels = next(train_generator)   # one batch

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
label_names = {v: k for k, v in train_generator.class_indices.items()}

for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i])
    ax.set_title(label_names[int(round(labels[i]))], fontsize=12)
    ax.axis("off")

fig.suptitle("Augmented Training Samples (normalised to [0,1])", fontsize=14)
fig.tight_layout()
out_path = os.path.join(PLOTS_DIR, "augmented_samples.png")
plt.savefig(out_path, dpi=150)
plt.close()
print(f"Augmented samples grid saved → {out_path}")


Augmented samples grid saved → D:\Hams Khaled\CS Year 3 Sem 2\Image Processing\Final Project\project\results\plots\augmented_samples.png


## Step 5 — Final Dataset Summary

In [10]:
print("=" * 50)
print("  FINAL DATASET SUMMARY")
print("=" * 50)
print(f"  Image size           : {IMG_SIZE[0]}x{IMG_SIZE[1]} (RGB)")
print(f"  Batch size           : {BATCH_SIZE}")
print(f"  Class indices        : {train_generator.class_indices}")
print()
for split, gen in [("Train", train_generator), ("Val", val_generator), ("Test", test_generator)]:
    print(f"  {split:<6}: {gen.samples:5d} images  |  {math.ceil(gen.samples/BATCH_SIZE)} batches")
print(f"\n  Grand total          : {train_generator.samples + val_generator.samples + test_generator.samples} images")
print("=" * 50)
print("Preprocessing complete!")


  FINAL DATASET SUMMARY
  Image size           : 224x224 (RGB)
  Batch size           : 32
  Class indices        : {'fresh': 0, 'rotten': 1}

  Train :  9518 images  |  298 batches
  Val   :  2040 images  |  64 batches
  Test  :  2041 images  |  64 batches

  Grand total          : 13599 images
Preprocessing complete!
